# M29 lab solution: Out-of-corpus stress test

**Brief.** Send questions the corpus cannot answer and compare the baseline, CRAG and Self-RAG on refusals, web fallbacks and hallucinated answers.

The 15 questions are the golden set's 5 out-of-corpus questions plus the 10 in `../data/ooc_questions.jsonl`. CRAG follows lesson 29.1 (grade the retrieved passages, fall back to a web search when none is relevant); Self-RAG follows lesson 29.2 (decide whether to retrieve, filter by relevance, check groundedness and usefulness). Larkspur is fictional, so the web holds nothing about it, and any direct answer to these questions is unsupported by the handbook.

The notebook reads only `../data/` and imports only from `src/`. Run it from this folder; the lab README lists the keys it needs.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
sys.path.insert(0, str(Path("../../../../src").resolve()))

from helpers import get_embeddings, get_experientiallabs_llm
from lab_kit import is_hit, load_handbook, load_jsonl

DATA = Path("../data")
llm = get_experientiallabs_llm()
print("LLM:", llm.model_name)

## The Part V system, rebuilt inline

This lab does not depend on another lab's output, so the cell below rebuilds the minimal Part V system: the handbook loaded with `lab_kit`, the baseline chunker (500 characters, 50 overlap), an in-memory vector store over `get_embeddings()`, and an answer prompt that cites passages by number. Later cells reuse `retriever`, `format_docs` and `rag_answer`.

In [ ]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

handbook = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [Document(page_content=c, metadata={"source": p["source"]}) for p in handbook for c in splitter.split_text(p["text"])]
embeddings = get_embeddings()
store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about the Larkspur Cloud handbook. Use only the numbered passages. "
               "Cite each claim with its passage number in square brackets, like [2]. "
               "If the passages do not contain the answer, reply: The handbook does not say."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])
answer_chain = ANSWER_PROMPT | llm | StrOutputParser()


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata['source']}) {d.page_content}" for i, d in enumerate(docs, 1))


def rag_answer(question, docs=None):
    """Retrieve (unless docs are given) and answer with citations; returns (answer, docs)."""
    docs = retriever.invoke(question) if docs is None else docs
    return answer_chain.invoke({"context": format_docs(docs), "question": question}), docs


print(len(chunks), "chunks;", len(golden), "golden questions")

In [ ]:
stress = [g for g in golden if g["type"] == "out_of_corpus"] + load_jsonl(DATA / "ooc_questions.jsonl")
print(len(stress), "out-of-corpus questions")

## 1. Shared checks: a per-passage grader and a web search

In [ ]:
from typing import Literal

from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field


class DocGrade(BaseModel):
    relevant: bool = Field(description="True if this passage helps answer the question")


doc_grader = ChatPromptTemplate.from_messages([
    ("system", "Say whether the passage helps answer the question. A passage on a related topic that does not answer it does not help."),
    ("human", "Passage:\n{passage}\n\nQuestion: {question}"),
]) | llm.with_structured_output(DocGrade)

web = TavilySearch(max_results=3)


def web_docs(query):
    results = web.invoke({"query": query}).get("results", [])
    return [Document(page_content=r["content"], metadata={"source": r["url"]}) for r in results]


def relevant_docs(question, docs):
    return [d for d in docs if doc_grader.invoke({"passage": d.page_content, "question": question}).relevant]

## 2. Three systems

Each returns the answer and a flag for whether it used the web.

In [ ]:
REFUSAL = "The handbook does not say."


def baseline_system(question):
    text, _ = rag_answer(question)
    return text, False


WEB_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Answer the question about Larkspur Cloud from these web results only, citing the URL. "
               "If they do not answer it, reply: I could not find this."),
    ("human", "Web results:\n{context}\n\nQuestion: {question}"),
])
web_chain = WEB_PROMPT | llm | StrOutputParser()


def crag_system(question):
    good = relevant_docs(question, retriever.invoke(question))
    if good:
        text, _ = rag_answer(question, good)
        return text, False
    docs = web_docs(f"Larkspur Cloud {question}")
    return web_chain.invoke({"context": format_docs(docs), "question": question}), True


class Decision(BaseModel):
    decision: Literal["yes", "no"]


retrieve_gate = ChatPromptTemplate.from_messages([
    ("system", "Decide whether answering this question needs the Larkspur Cloud handbook. Answer yes or no."),
    ("human", "{question}"),
]) | llm.with_structured_output(Decision)
grounded_check = ChatPromptTemplate.from_messages([
    ("system", "Is every claim in the answer supported by the passages? Answer yes or no."),
    ("human", "Passages:\n{context}\n\nAnswer:\n{answer}"),
]) | llm.with_structured_output(Decision)
useful_check = ChatPromptTemplate.from_messages([
    ("system", "Does the answer address the question? Answer yes or no."),
    ("human", "Question: {question}\n\nAnswer:\n{answer}"),
]) | llm.with_structured_output(Decision)


def self_rag_system(question):
    if retrieve_gate.invoke({"question": question}).decision == "no":
        return llm.invoke(question).content, False  # Self-RAG may answer without retrieval
    good = relevant_docs(question, retriever.invoke(question))
    if not good:
        return REFUSAL, False
    for _ in range(2):  # one regeneration when the first draft is not grounded
        text, _ = rag_answer(question, good)
        if grounded_check.invoke({"context": format_docs(good), "answer": text}).decision == "yes":
            break
    else:
        return REFUSAL, False
    if useful_check.invoke({"question": question, "answer": text}).decision == "no":
        return REFUSAL, False
    return text, False


SYSTEMS = {"baseline": baseline_system, "CRAG": crag_system, "Self-RAG": self_rag_system}

## 3. The judge: a refusal or an answer

The handbook answers none of these questions, so a direct answer is a hallucinated answer, whether it came from the model or from an unrelated web page. The judge only has to tell an answer from a refusal.

In [ ]:
class Verdict(BaseModel):
    label: Literal["refusal", "answer"]
    reason: str


JUDGE_SYSTEM = ("Classify the reply. 'refusal': it says the information is not available, not found or not in the handbook, "
                "even if it adds related facts. 'answer': it gives a direct answer to the question (a price, a yes or no, a name, "
                "a number, a date, a list), whatever its source.")
judge = ChatPromptTemplate.from_messages([
    ("system", JUDGE_SYSTEM),
    ("human", "Question: {question}\n\nReply:\n{reply}"),
]) | llm.with_structured_output(Verdict)
print("Judge prompt (system message):\n" + JUDGE_SYSTEM)

rows = []
for g in stress:
    for name, system in SYSTEMS.items():
        reply, used_web = system(g["question"])
        v = judge.invoke({"question": g["question"], "reply": reply})
        rows.append({"id": g["id"], "system": name, "web fallback": used_web, "label": v.label,
                     "hallucinated": v.label == "answer", "reply": reply})
log = pd.DataFrame(rows)
print(log.pivot(index="id", columns="system", values="label").to_string())

## 4. Results

In [ ]:
summary = log.groupby("system").agg(questions=("id", "nunique"), refusals=("label", lambda s: (s == "refusal").sum()),
                                    web_fallbacks=("web fallback", "sum"), hallucinated=("hallucinated", "sum"))
print(summary.loc[list(SYSTEMS)].to_string())
print("\nHallucinated replies:")
for _, r in log[log.hallucinated].iterrows():
    print(f"  {r.id} [{r.system}{', web' if r['web fallback'] else ''}] {r.reply[:200]!r}")

## What this shows

- The summary has the three counts the brief asks for, per system, on the same 15 questions.
- CRAG's web fallback is a design choice with a cost here: for a fictional company the web can only return pages about something else, so a fallback either finds nothing or answers from an unrelated source. Read its hallucinated replies to see which.
- Self-RAG's first gate can skip retrieval altogether, and a reply without retrieval has nothing to be grounded in. Its other refusals come from the relevance filter, the groundedness check and the usefulness check; the trace of which check fired is in the function, if you want to log it.